# Feature Engineering - The Breakthrough
## Based on EDA Findings
**Thesis:** Novel representations that drastically simplify the learning task.

## 1. Setup

In [ ]:
import warnings
import numpy as np, pandas as pd
from scipy.stats import pearsonr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')
np.random.seed(42)

## 2. Load Real Data

In [ ]:
from datasets import load_dataset
ds = load_dataset('nkazi/MohlerASAG')
records = []
for split in ds.keys():
    for row in ds[split]:
        if not row.get('id'): continue
        q_id = '.'.join(row['id'].split('.')[:2])
        records.append({
            'question_id': q_id,
            'answer': str(row['student_answer']).replace('<br>','').strip(),
            'score': float(row.get('score_avg', 0))
        })
df = pd.DataFrame(records)
df['word_count'] = df['answer'].str.split().str.len()
df['label'] = (df['score'] >= 3).astype(int)

# TF-IDF cosine similarity to reference
ref_answers = df.groupby('question_id')['answer'].first().to_dict()
def get_tfidf_sim(row):
    ref = ref_answers.get(row['question_id'], '')
    if not ref: return np.nan
    try:
        vec = TfidfVectorizer().fit_transform([ref, row['answer']])
        return float(cosine_similarity(vec[0], vec[1])[0,0])
    except: return np.nan
df['tfidf_sim'] = df.apply(get_tfidf_sim, axis=1)
print(f"Data: {len(df)} samples")

## 3. Feature 1: Similarity Margin
**EDA Driver:** Overlap zone where correct/incorrect have similar TF-IDF
**Solution:** Measure how much better than typical wrong answer

In [ ]:
q_wrong_mean = df[df['label'] == 0].groupby('question_id')['tfidf_sim'].mean()
df = df.merge(q_wrong_mean.rename('wrong_mean').reset_index(), on='question_id', how='left')
df['wrong_mean'] = df['wrong_mean'].fillna(df['tfidf_sim'].mean() * 0.3)
df['feat_margin'] = df['tfidf_sim'] - df['wrong_mean']

# Quantify improvement
hi_raw = df[df['label']==1]['tfidf_sim'].values
lo_raw = df[df['label']==0]['tfidf_sim'].values
hi_mar = df[df['label']==1]['feat_margin'].values
lo_mar = df[df['label']==0]['feat_margin'].values

d_raw = (hi_raw.mean() - lo_raw.mean()) / np.sqrt((hi_raw.std()**2 + lo_raw.std()**2)/2)
d_mar = (hi_mar.mean() - lo_mar.mean()) / np.sqrt((hi_mar.std()**2 + lo_mar.std()**2)/2)

f1_base = max([f1_score(df['label'], (df['tfidf_sim'] >= t).astype(int), zero_division=0) for t in np.arange(-0.5, 0.8, 0.01)])
f1_mar = max([f1_score(df['label'], (df['feat_margin'] >= t).astype(int), zero_division=0) for t in np.arange(-0.5, 0.8, 0.01)])

print(f'Feature 1 - Similarity Margin:')
print(f"  Cohen's d: {d_raw:.3f} -> {d_mar:.3f}")
print(f"  F1: {f1_base:.4f} -> {f1_mar:.4f} (+{f1_mar-f1_base:.4f})")

## 4. Feature 2: Length-Decoupled Similarity
**EDA Driver:** Length confounds grading (short correct answers penalized)
**Solution:** Remove length effect via regression residual

In [ ]:
df['log_wc'] = np.log1p(df['word_count'])
lr = LinearRegression().fit(df[['log_wc']], df['tfidf_sim'])
df['feat_lds'] = df['tfidf_sim'] - lr.predict(df[['log_wc']])

r_before, _ = pearsonr(df['word_count'], df['tfidf_sim'])
r_after, _ = pearsonr(df['word_count'], df['feat_lds'])
f1_lds = max([f1_score(df['label'], (df['feat_lds'] >= t).astype(int), zero_division=0) for t in np.arange(-0.5, 0.8, 0.01)])

print(f'Feature 2 - Length-Decoupled Similarity:')
print(f"  Word count correlation: r={r_before:.3f} -> {r_after:.3f}")
print(f"  F1: {f1_base:.4f} -> {f1_lds:.4f} (+{f1_lds-f1_base:.4f})")

## 5. Feature 3: Question-Difficulty Residual
**EDA Driver:** Same similarity means different things on hard vs easy questions
**Solution:** Normalize by question-level baseline

In [ ]:
q_correct_mean = df[df['label'] == 1].groupby('question_id')['tfidf_sim'].mean()
df = df.merge(q_correct_mean.rename('q_correct_mean').reset_index(), on='question_id', how='left')
df['q_correct_mean'] = df['q_correct_mean'].fillna(df['tfidf_sim'].mean())
df['feat_qdr'] = df['tfidf_sim'] - df['q_correct_mean']

f1_qdr = max([f1_score(df['label'], (df['feat_qdr'] >= t).astype(int), zero_division=0) for t in np.arange(-0.5, 0.5, 0.01)])

print(f'Feature 3 - Question-Difficulty Residual:')
print(f"  F1: {f1_base:.4f} -> {f1_qdr:.4f} (+{f1_qdr-f1_base:.4f})")

## 6. Ablation Study

In [ ]:
FEATURES = ['tfidf_sim', 'feat_margin', 'feat_lds', 'feat_qdr']
X = df[FEATURES].fillna(0).values
y = df['label'].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\n{'Feature Set':<25} {'F1':>6}")
print('-'*35)

f1_base_cv = cross_val_score(Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression())]), X[:,:1], y, cv=cv, scoring='f1').mean()
print(f"{'TF-IDF baseline':<25} {f1_base_cv:.4f}")

f1_full = cross_val_score(Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression())]), X, y, cv=cv, scoring='f1').mean()
print(f"{'All features':<25} {f1_full:.4f}")

for i, f in enumerate(FEATURES[1:], 1):
    cols = [c for c in range(len(FEATURES)) if c != i]
    f1_abl = cross_val_score(Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression())]), X[:, cols], y, cv=cv, scoring='f1').mean()
    print(f"{'without ' + f:<25} {f1_abl:.4f} (drop: {f1_full-f1_abl:+.4f})")